# 1. Initialization

In [0]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(repo_root)

print(f"[INFO] Repo root added to path: {repo_root}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col, to_timestamp, lit, upper, trim, create_map
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable
from functools import reduce
from itertools import chain

from common.helpers import get_bronze

print("[INFO] Payments Silver pipeline started")

try:
    dbutils.widgets.text("batch_id", "")
    batch_id = dbutils.widgets.get("batch_id")
except:
    batch_id = None

print("[INFO] Databricks Spark session ready")
spark
print(f"[INFO] Batch ID: {batch_id}")

# 2. Read Bronze Payments

In [0]:
payments_bronze = "/Volumes/datalake_catalog/datalake_schema/bronze/payments"
df_bronze_payments = get_bronze(payments_bronze, spark=spark)
df_bronze_payments.show(5)

# 3. Remove duplicates (business grain)

In [0]:
df1 = df_bronze_payments.drop("dw_ingested_at", "dw_source_file", "dw_batch_id", "batch_id", "source_table")

df_payments_clean = (
    df1
    .withColumn("payment_date", F.to_timestamp("payment_date"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time"))
)

window_spec = Window.partitionBy("payment_id", "payment_date").orderBy(F.col("ingest_time").desc())

df_with_rn = df_payments_clean.withColumn("rn", F.row_number().over(window_spec))

df2 = df_with_rn.filter(F.col("rn") == 1).drop("rn")
df2_quarantine = df_with_rn.filter(F.col("rn") > 1).drop("rn")

print("Valid:", df2.count(), "Quarantine:", df2_quarantine.count())

# 4. Cast Data Types

In [0]:
df3 = df2.select(
    col("payment_id").cast("bigint"),
    col("subscription_id").cast("bigint"),
    col("amount").cast("decimal(10,2)"),
    col("currency"),
    col("payment_status"),
    to_timestamp(col("payment_date")).alias("payment_date"),
    col("payment_method"),
    to_timestamp(col("ingest_time")).alias("ingest_time")
)

# 5. Normalize & Validate Categorical Data

In [0]:
valid_payment_status = ["refunded", "success", "failed"]
valid_currency = ["GBP", "EUR", "USD", "VND"]
valid_payment_method = ["google_pay","paypal","apple_pay","credit_card","bank_transfer","invoice"]

df_payments_norm = (
    df3
    .withColumn("payment_status", F.lower(trim(col("payment_status"))))
    .withColumn("currency", F.upper(trim(col("currency"))))
    .withColumn("payment_method", F.lower(trim(col("payment_method"))))
)

df4 = df_payments_norm.filter(
    col("payment_status").isin(valid_payment_status) &
    col("currency").isin(valid_currency) &
    col("payment_method").isin(valid_payment_method)
)

# 6. Referential Integrity Check

In [0]:
df_sub = spark.read.format("delta").load("/Volumes/datalake_catalog/datalake_schema/silver/subscriptions")
sub_ref = df_sub.select("subscription_id").dropDuplicates()

df5 = df4.join(sub_ref, "subscription_id", "inner")
df5_quarantine = df4.join(sub_ref, "subscription_id", "left_anti")

# 7. Currency Conversion

In [0]:
rates = {"USD":1.0,"EUR":0.93,"GBP":0.80,"VND":25500.0}
rate_map = create_map([lit(x) for x in chain(*rates.items())])

df6 = (
    df5
    .withColumn("exchange_rate", rate_map[col("currency")].cast(DecimalType(18,6)))
    .withColumn("amount_usd", (col("amount") * col("exchange_rate")).cast(DecimalType(18,2)))
)

# 8. Write to Silver Delta Lake

In [0]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/payments"

df_final = df6.dropDuplicates(["payment_id", "payment_date"])

if DeltaTable.isDeltaTable(spark, silver_path):
    target = DeltaTable.forPath(spark, silver_path)
    
    target.alias("t").merge(
        df_final.alias("s"),
        "t.payment_id = s.payment_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("overwrite").save(silver_path)

print("[SUCCESS] Payments written to Silver Delta")

# 9. Completed

In [0]:
print("[DONE] Payments Silver pipeline completed")